# 01 - Document Loading - How do we reliably load documents?

## Objective

Evaluate approaches for loading the project knowledge base into the RAG pipeline.

The objective is to identify a simple, reliable, and maintainable document loading strategy suitable for the project's current scope.

This notebook focuses on experimentation rather than production implementation.

---

## Questions

- Which document formats should be supported?
- Which document loader best preserves the original content?
- Is the extracted text suitable for chunking?
- Which metadata should be retained?
- Are any preprocessing steps required before chunking?

---

## Success Criteria

By the end of this notebook:

- Documents can be loaded successfully.
- Extracted text is clean and readable.
- Relevant metadata is available.
- A document loading strategy has been selected for implementation.

---

## Notes

This notebook is part of the experimentation phase.

Once the approach is validated, the implementation will be refactored into the production codebase under `src/`.

In [49]:
from dotenv import load_dotenv
import os
import glob 
import numpy as np 
from pathlib import Path
import re

from langchain_core.documents import Document
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyMuPDFLoader, Docx2txtLoader

In [2]:
load_dotenv(override=True)  # take environment variables from .env.
openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable not set.")

## Project Paths and Knowledge base directory

In [4]:
# project paths 
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw/knowledge_base"

In [33]:
# Inspect available files in the knowledge base directory
files = glob.glob(str(RAW_DIR / "**/*"))
print("Available files in the knowledge base directory:")

# entire_knowledge_base = ''

for file in files:
    print(f" - {Path(file).name}")
#     with open(file, 'r', encoding='utf-8') as f:
#         entire_knowledge_base += f.read() + "\n\n"  # Add a newline between files for separation

# print(f'Total characters in the entire knowledge base: {len(entire_knowledge_base):,}')

Available files in the knowledge base directory:
 - quick_service_restaurant.md
 - CoverLetter_DR_4.pdf
 - CoverLetter_DR_2.pdf
 - CoverLetter_DR_3.pdf
 - CoverLetter_DR_1.pdf
 - CV_DR_FS.docx


## Load one document

### PDF format

In [34]:
loader = PyMuPDFLoader(RAW_DIR / "cover_letters/CoverLetter_DR_1.pdf")
documents = loader.load()
# print(documents[0].metadata)

In [36]:
# inspect the resulting document
print(documents[0].page_content[130:1000]) # this is shwowing breaklines and other formatting issues. We will need to clean this up before using it in a prompt.

 COVER LETTER                                     
 
 
             .            
 May 2026 
 
Dear Hiring Team, 
 
I am excited to apply for the Senior Manager - Strategic Analytics & Testing position at American 
Express. After going through the job description, I am confident that my technical skills combined with 
my consulting and commercial background are a strong fit for empowering the marketing team and 
different stakeholders through commercially relevant insights. I am a Principal Data Scientist with 5+ 
years of experience in consulting delivering advanced analytics projects in the industries of Financial 
Services and Retail. Further, I complement my technical experience with 4 additional years in finance 
and business analysis roles that have given me the basis for focusing on highly relevant, commercially 
sounding insights and modelling.  
 
I


### .docx format

In [37]:
loader = Docx2txtLoader(RAW_DIR / "cv/CV_DR_FS.docx")
documents = loader.load()
print(documents[0].metadata)

{'source': '/Users/davidr/git_projects/ai-career-assistant/data/raw/knowledge_base/cv/CV_DR_FS.docx'}


In [41]:
print(documents[0].page_content[350:1000]) 


  PROFESSIONAL PROFILE                           				        .                                               

Principal Data Scientist with 5+ years of experience in consulting delivering advanced analytics, BI, and machine learning solutions in the industries of Financial Services and Retail. I complement my technical experience with 4 additional years in finance and business analysis roles. I have a strong focus on solving commercial problems end-to-end, from framing business questions with senior stakeholders to building ML models that solve real world problems. I bring advanced proficiency in Python and SQL, which allow me to face ambig


## Load every document

In [46]:
LOADERS = {".pdf": PyMuPDFLoader, ".docx": Docx2txtLoader, ".md": TextLoader, ".txt": TextLoader}

documents = []
for path in sorted(RAW_DIR.rglob("*")):
    if path.is_dir():
        continue
    loader_cls = LOADERS.get(path.suffix.lower())
    if loader_cls is None:
        print(f"Skipping unsupported file: {path.name}")
        continue
    loader = loader_cls(str(path))
    documents.extend(loader.load())

print(f"Loaded {len(documents)} documents.")

Skipping unsupported file: .DS_Store
Skipping unsupported file: .DS_Store
Loaded 6 documents.


### Clean line breaks
In the examples from the previous section it was observed that the document loaders can add line breaks for some of the sentences (pdf) or paragraphs. 

In [47]:
def clean_text(text):
    # Collapse single newlines (likely mid-sentence wraps) into spaces,
    # but preserve intentional paragraph breaks (blank lines).
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # Collapse 3+ newlines down to a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Collapse repeated spaces/tabs.
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text.strip()

In [50]:
for doc in documents:
    doc.page_content = clean_text(doc.page_content)

## Conclusion

### Decision

Use a `LOADER` registry to handle different document types for document ingestion.

### Rationale

- The knowledge base can have a combination of .docx, .pdf, .md, .txt documents
- For this version, leaving any additional document types out of the following sections

### Next Step

Proceed to document chunking.